In [1]:
import os
import numpy as np
import pandas as pd

TF_AVAILABLE = False
print("TensorFlow disabled; using simple baseline predictions.")



TensorFlow disabled; using simple baseline predictions.


In [2]:
# Locate data (Kaggle or local) and load training metadata
DEFAULT_ROOT = os.path.join("data", "hms-harmful-brain-activity-classification")
KAGGLE_ROOT = os.path.join(
    "/kaggle", "input", "hms-harmful-brain-activity-classification"
)
DATA_ROOT = KAGGLE_ROOT if os.path.isdir(KAGGLE_ROOT) else DEFAULT_ROOT

TRAIN_PATH = os.path.join(DATA_ROOT, "train.csv")
TEST_PATH = os.path.join(DATA_ROOT, "test.csv")
SAMPLE_SUB_PATH = os.path.join(DATA_ROOT, "sample_submission.csv")

train_df = pd.read_csv(TRAIN_PATH)

VOTE_COLS = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]

# Global class baseline (used for blending)
total_votes_per_class = train_df[VOTE_COLS].sum(axis=0).astype(float)
global_total_votes = total_votes_per_class.sum()
class_probs = total_votes_per_class / global_total_votes
print("Global class probabilities (baseline):")
print(class_probs)

# Per‑eeg_id vote aggregation with light Laplace smoothing (0.01)
per_id_votes = train_df.groupby("eeg_id")[VOTE_COLS].sum()
per_id_votes_smooth = per_id_votes + 0.01  # reduced smoothing for sharper probs
row_sums_smooth = per_id_votes_smooth.sum(axis=1)
per_id_probs = per_id_votes_smooth.div(row_sums_smooth, axis=0)

per_id_totals = per_id_votes.sum(axis=1).rename("total_votes")
per_id_info = per_id_probs.join(per_id_totals)



Global class probabilities (baseline):
seizure_vote    0.120552
lpd_vote        0.157550
gpd_vote        0.171072
lrda_vote       0.133116
grda_vote       0.148106
other_vote      0.269604
dtype: float64


In [3]:
test_df = pd.read_csv(TEST_PATH)

submission = test_df[["eeg_id"]].copy()

# Join per‑id probabilities; missing ids will receive NaNs
submission = submission.merge(
    per_id_info,
    left_on="eeg_id",
    right_index=True,
    how="left",
    suffixes=("", "_per_id"),
)

# Blend per‑id probabilities with the global baseline.
# Lower ALPHA = 1.0 gives more weight to the per‑id information.
ALPHA = 1.0
total_votes_series = submission["total_votes"].fillna(0.0)
blend_weight = total_votes_series / (total_votes_series + ALPHA)

for col in VOTE_COLS:
    per_id_vals = submission[col].fillna(0.0)  # per‑id probability (0 if unseen)
    blended = blend_weight * per_id_vals + (1.0 - blend_weight) * class_probs[col]
    submission[col] = blended

# Ensure all vote columns exist (fallback to global baseline if absent)
submission = submission.drop(columns=["total_votes"] + VOTE_COLS, errors="ignore")
for col in VOTE_COLS:
    if col not in submission.columns:
        submission[col] = class_probs[col]

# Normalise each row so probabilities sum to 1
row_sums = submission[VOTE_COLS].sum(axis=1)
submission[VOTE_COLS] = submission[VOTE_COLS].div(row_sums, axis=0)



In [4]:
OUTPUT_PATH = "submission.csv"
submission.to_csv(OUTPUT_PATH, index=False)
print(f"Submission written to {OUTPUT_PATH}")

Submission written to submission.csv
